# Constrained Decoding Evaluation

Use this notebook to evaluate a trained reduced-schema adapter with schema-constrained decoding.

Recommended default:

- checkpoint: `results/checkpoints/qwen25_3b_stage2_epoch9_rank16_full`
- schema: `ticket_schema_v1_reduced`
- test file: `data/reduced/phase1_test_reduced.jsonl`
- output prefix: `qwen25_3b_stage2_epoch9_rank16_constrained_v1`

This notebook writes raw predictions, raw evaluation, field analysis, repaired predictions, and repaired evaluation in the same format as the Stage 2 runner.

In [1]:
# Uncomment this once in a fresh online environment if needed.
# %pip install -q transformers datasets peft accelerate bitsandbytes pyyaml jsonschema lm-format-enforcer

In [2]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import torch


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not find project root')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('project_root =', PROJECT_ROOT)
print('python =', sys.version)
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device =', torch.cuda.get_device_name(0))
    print('bf16_supported =', torch.cuda.is_bf16_supported())

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Compatibility patch for lm-format-enforcer with newer transformers layouts.
import transformers.tokenization_utils as tokenization_utils
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
if not hasattr(tokenization_utils, 'PreTrainedTokenizerBase'):
    tokenization_utils.PreTrainedTokenizerBase = PreTrainedTokenizerBase

from lmformatenforcer import JsonSchemaParser
from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn

from src.data.io import dump_jsonl, load_jsonl
from src.evaluation.field_analysis import analyze_field_errors
from src.evaluation.metrics import evaluate_sample, summarize_results, try_parse_prediction_text
from src.evaluation.reporting import group_sample_results, write_json_report
from src.inference.repair import repair_prediction
from src.schemas.registry import get_schema
from src.training.formatters import DEFAULT_SYSTEM_PROMPT, build_user_prompt

project_root = /home/lyan11/small-llm-structured-posttraining
python = 3.13.11 | packaged by conda-forge | (main, Dec  6 2025, 11:24:03) [GCC 14.3.0]
cuda_available = True
device = NVIDIA H200 NVL
bf16_supported = True


In [3]:
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
CHECKPOINT_DIR = PROJECT_ROOT / 'results' / 'checkpoints' / 'qwen25_3b_stage2_epoch9_rank16_full'
TEST_PATH = PROJECT_ROOT / 'data' / 'reduced' / 'phase1_test_reduced.jsonl'
SCHEMA_NAME = 'ticket_schema_v1_reduced'
EXPERIMENT_NAME = 'qwen25_3b_stage2_epoch9_rank16_constrained_v1'
MAX_NEW_TOKENS = 256

assert CHECKPOINT_DIR.exists(), f'Missing checkpoint dir: {CHECKPOINT_DIR}'
assert TEST_PATH.exists(), f'Missing test file: {TEST_PATH}'

print('base_model =', BASE_MODEL)
print('checkpoint_dir =', CHECKPOINT_DIR)
print('test_path =', TEST_PATH)
print('schema_name =', SCHEMA_NAME)
print('experiment_name =', EXPERIMENT_NAME)

base_model = Qwen/Qwen2.5-3B-Instruct
checkpoint_dir = /home/lyan11/small-llm-structured-posttraining/results/checkpoints/qwen25_3b_stage2_epoch9_rank16_full
test_path = /home/lyan11/small-llm-structured-posttraining/data/reduced/phase1_test_reduced.jsonl
schema_name = ticket_schema_v1_reduced
experiment_name = qwen25_3b_stage2_epoch9_rank16_constrained_v1


In [4]:
schema = get_schema(SCHEMA_NAME)


def sanitize_schema_for_lmfe(node):
    if isinstance(node, dict):
        updated = {key: sanitize_schema_for_lmfe(value) for key, value in node.items()}
        enum_values = updated.get('enum')
        if isinstance(enum_values, list) and any(value is None for value in enum_values):
            updated['enum'] = [value for value in enum_values if value is not None]
        return updated
    if isinstance(node, list):
        return [sanitize_schema_for_lmfe(value) for value in node]
    return node


decode_schema = sanitize_schema_for_lmfe(schema)
test_records = load_jsonl(TEST_PATH)
print('num_test_records =', len(test_records))

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, str(CHECKPOINT_DIR))
model.eval()

print('pad_token =', tokenizer.pad_token)
print('decode schema sanitized for lm-format-enforcer enum compatibility')
print('model loaded with adapter checkpoint')

num_test_records = 254


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

pad_token = <|endoftext|>
decode schema sanitized for lm-format-enforcer enum compatibility
model loaded with adapter checkpoint


In [5]:
generation_kwargs = {
    'max_new_tokens': MAX_NEW_TOKENS,
    'do_sample': False,
    'temperature': 1.0,
    'top_p': 1.0,
}


def build_inference_messages(record):
    return [
        {'role': 'system', 'content': DEFAULT_SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': build_user_prompt(
                input_text=record['input_text'],
                task_name=record['task_name'],
                schema_name=record['schema_name'],
                include_schema_definition=False,
            ),
        },
    ]


def generate_prediction_text(record):
    parser = JsonSchemaParser(decode_schema)
    prefix_allowed_tokens_fn = build_transformers_prefix_allowed_tokens_fn(tokenizer, parser)
    prompt_text = tokenizer.apply_chat_template(
        build_inference_messages(record),
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        **generation_kwargs,
        prefix_allowed_tokens_fn=prefix_allowed_tokens_fn,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


sample_pred = generate_prediction_text(test_records[0])
print(sample_pred)

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{"summary": "Password Reset Assistance Needed", "category": "task", "priority": "medium", "requires_followup": true, "affected_systems": [{"name": "Account", "component": "account"}], "actions_requested": [{"action": "Handle request: Password Reset Assistance Needed", "owner": null, "deadline": null}], "constraints": {"environment": "prod", "blocking": null}}


In [6]:
predictions = []
for idx, record in enumerate(test_records):
    prediction_text = generate_prediction_text(record)
    try:
        prediction_json = json.loads(prediction_text)
    except json.JSONDecodeError:
        prediction_json = None
    predictions.append(
        {
            'sample_id': record['sample_id'],
            'prediction_text': prediction_text,
            'prediction_json': prediction_json,
            'metadata': {
                'model_name': BASE_MODEL,
                'experiment_id': EXPERIMENT_NAME,
                'decode_mode': 'schema_constrained',
            },
        }
    )
    if (idx + 1) % 25 == 0:
        print(f'generated {idx + 1} / {len(test_records)}')

prediction_path = PROJECT_ROOT / 'results' / 'predictions' / f'{EXPERIMENT_NAME}_test.jsonl'
dump_jsonl(prediction_path, predictions)
print('prediction_path =', prediction_path)
print('num_predictions =', len(predictions))

generated 25 / 254
generated 50 / 254
generated 75 / 254
generated 100 / 254
generated 125 / 254
generated 150 / 254
generated 175 / 254
generated 200 / 254
generated 225 / 254
generated 250 / 254
prediction_path = /home/lyan11/small-llm-structured-posttraining/results/predictions/qwen25_3b_stage2_epoch9_rank16_constrained_v1_test.jsonl
num_predictions = 254


In [7]:
def sample_eval_dicts(gold_records, pred_records):
    predictions_by_id = {record['sample_id']: record for record in pred_records}
    results = []
    for gold_record in gold_records:
        pred_record = predictions_by_id.get(gold_record['sample_id'], {})
        sample_eval = evaluate_sample(
            sample_id=gold_record['sample_id'],
            prediction_text=pred_record.get('prediction_text'),
            prediction_json=pred_record.get('prediction_json'),
            target_json=gold_record['target_json'],
            schema=schema,
        )
        results.append(
            {
                **sample_eval.__dict__,
                'schema_name': gold_record['schema_name'],
                'complexity_bucket': gold_record.get('complexity_bucket', 'unknown'),
            }
        )
    return results


def summarize_from_dicts(sample_results):
    from src.evaluation.metrics import SampleEvaluation

    evaluations = [
        SampleEvaluation(
            sample_id=item['sample_id'],
            valid_json=item['valid_json'],
            schema_compliant=item['schema_compliant'],
            field_exact_match=item['field_exact_match'],
            exact_match=item['exact_match'],
            primary_error=item['primary_error'],
        )
        for item in sample_results
    ]
    return summarize_results(evaluations)


raw_sample_results = sample_eval_dicts(test_records, predictions)
raw_report = {
    'summary': summarize_from_dicts(raw_sample_results),
    'grouped_summary': {
        'by_complexity_bucket': {
            name: summarize_from_dicts(items)
            for name, items in group_sample_results(raw_sample_results, 'complexity_bucket').items()
        },
    },
    'per_sample': raw_sample_results,
}

raw_report_path = PROJECT_ROOT / 'results' / 'metrics' / f'{EXPERIMENT_NAME}_test_report.json'
raw_field_path = PROJECT_ROOT / 'results' / 'metrics' / f'{EXPERIMENT_NAME}_field_analysis.json'
write_json_report(raw_report_path, raw_report)
write_json_report(raw_field_path, analyze_field_errors(test_records, predictions))

repaired_predictions = []
for record in predictions:
    prediction_json = record.get('prediction_json')
    prediction_text = record.get('prediction_text')
    if prediction_json is None and isinstance(prediction_text, str):
        _, prediction_json = try_parse_prediction_text(prediction_text)
    repaired_json, repaired = repair_prediction(prediction_json, schema)
    repaired_predictions.append(
        {
            **record,
            'prediction_json': repaired_json,
            'metadata': {
                **record.get('metadata', {}),
                'repaired': repaired,
            },
        }
    )

repaired_sample_results = sample_eval_dicts(test_records, repaired_predictions)
repaired_report = {
    'summary': summarize_from_dicts(repaired_sample_results),
    'per_sample': repaired_sample_results,
}

repaired_pred_path = PROJECT_ROOT / 'results' / 'predictions' / f'{EXPERIMENT_NAME}_test_repaired.jsonl'
repaired_report_path = PROJECT_ROOT / 'results' / 'metrics' / f'{EXPERIMENT_NAME}_test_repaired_report.json'
repaired_field_path = PROJECT_ROOT / 'results' / 'metrics' / f'{EXPERIMENT_NAME}_test_repaired_field_analysis.json'
dump_jsonl(repaired_pred_path, repaired_predictions)
write_json_report(repaired_report_path, repaired_report)
write_json_report(repaired_field_path, analyze_field_errors(test_records, repaired_predictions))

print('raw_report =', raw_report_path)
print('raw_field =', raw_field_path)
print('repaired_pred =', repaired_pred_path)
print('repaired_report =', repaired_report_path)
print('repaired_field =', repaired_field_path)

raw_report = /home/lyan11/small-llm-structured-posttraining/results/metrics/qwen25_3b_stage2_epoch9_rank16_constrained_v1_test_report.json
raw_field = /home/lyan11/small-llm-structured-posttraining/results/metrics/qwen25_3b_stage2_epoch9_rank16_constrained_v1_field_analysis.json
repaired_pred = /home/lyan11/small-llm-structured-posttraining/results/predictions/qwen25_3b_stage2_epoch9_rank16_constrained_v1_test_repaired.jsonl
repaired_report = /home/lyan11/small-llm-structured-posttraining/results/metrics/qwen25_3b_stage2_epoch9_rank16_constrained_v1_test_repaired_report.json
repaired_field = /home/lyan11/small-llm-structured-posttraining/results/metrics/qwen25_3b_stage2_epoch9_rank16_constrained_v1_test_repaired_field_analysis.json


In [8]:
print('raw_summary =')
print(json.dumps(raw_report['summary'], indent=2))
print()
print('repaired_summary =')
print(json.dumps(repaired_report['summary'], indent=2))

raw_summary =
{
  "num_samples": 254,
  "valid_json_rate": 0.9881889763779528,
  "schema_compliance_rate": 0.9881889763779528,
  "field_exact_match": 0.8056549749463134,
  "end_to_end_exact_match": 0.015748031496062992,
  "error_counts": {
    "invalid_json": 3,
    "schema_violation": 0,
    "missing_required_field": 0,
    "enum_error": 0,
    "type_error": 0,
    "value_hallucination": 247,
    "extraction_omission": 0,
    "cross_field_inconsistency": 0
  }
}

repaired_summary =
{
  "num_samples": 254,
  "valid_json_rate": 0.9881889763779528,
  "schema_compliance_rate": 0.9881889763779528,
  "field_exact_match": 0.8056549749463134,
  "end_to_end_exact_match": 0.015748031496062992,
  "error_counts": {
    "invalid_json": 3,
    "schema_violation": 0,
    "missing_required_field": 0,
    "enum_error": 0,
    "type_error": 0,
    "value_hallucination": 247,
    "extraction_omission": 0,
    "cross_field_inconsistency": 0
  }
}
